# Retrieval-Augmented Generation (RAG)

In [1]:
import chromadb
import dotenv
from pathlib import Path
from agents import Agent, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [2]:
# We populated the RAG with the data from the data/calories.csv file in
# the rag_setup.ipynb notebook

chroma_client = chromadb.PersistentClient("../chroma")
nutrition_db  = chroma_client.get_collection(name="nutrition_db")
nutrition_qna = chroma_client.get_collection(name="nutrition_qna")

In [3]:
results = nutrition_qna.query(query_texts=["orange"], n_results=2)
for i, doc in enumerate(results["documents"][0]):
    print(sorted(results["metadatas"][0][i].items()))
    print(doc)
    print("\n")

[('is_pregnancy', False)]
Question: What do yellow/orange veggies like squash have in common with each other?
        Answer: They all contain the same pigment called beta carotene.

        This Q&A pair provides information about nutrition and health topics.


[('is_pregnancy', False)]
Question: How can a protein-based meal with a citrus fruit impact one's morning routine?
        Answer: Consuming a protein source along with a citrus fruit at breakfast time can lead to improved focus, positive social interactions, reduced snacking tendencies, and increased feelings of well-being.

        This Q&A pair provides information about nutrition and health topics.




In [ ]:
@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

In [4]:
@function_tool
def nutrtition_qna_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function too ask a question about nutrition.

    Args:
        query: The question to ask
        max_results: The maximum number of results to return.

    Returns:
        A string containing the question and the answer related to the query.
    """

    results = nutrition_qna.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        formatted_results.append(doc)

    return "Related answers to your question:\n" + "\n".join(formatted_results)

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `calorie_lookup_tool` function_

In [ ]:
nutrtition_qna_tool('What is smart for a pregnant woman to eat when in first 3 months of pregnancy?')

In [5]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out information about pregnancy and nutrition while pregnant.
    You give concise answers.
    You need to get answers by using nutrtition_qna_tool.
    """,
    tools=[nutrtition_qna_tool],
)

In [6]:
with trace("Nutrition Assistant with RAG"):
    result = await Runner.run(
        calorie_agent,
        "What is smart for a pregnant woman to eat when in first 3 months of pregnancy?",
    )
    print(result.final_output)

Key smart choices for the first 3 months of pregnancy:

- Focus on folate (folic acid): leafy greens, fortified grains, beans; consider a prenatal vitamin with folate as directed by your provider.
- Iron-rich foods: lean meats, poultry, fish (low-mercury), eggs, fortified cereals, legumes; pair with vitamin C-rich foods to aid absorption.
- Calcium and vitamin D: dairy or fortified alternatives, leafy greens, small portions of fish with bones (e.g., sardines) if advised.
- Protein: lean meats, fish low in mercury, eggs, dairy, legumes, nuts, seeds.
- Omega-3 (DHA): fatty fish low in mercury (salmon, sardines) or a DHA supplement if recommended.
- Hydration and fiber: water, fruits/vegetables, whole grains to help digestion and prevent constipation.
- Iodine: include dairy, seafood (low-mercury) or iodized salt as advised.
- Avoid: raw/undercooked eggs or meats, unpasteurized dairy, high-mercury fish (shark, swordfish, king mackerel, tilefish), unwashed produce, alcohol, excessive caffe